In [524]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [525]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [526]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-908289896 | INFO | 2516037 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
,-908289835 | INFO | 2516037 - Setting seeds to: 0
!,-908289832 | INFO | 2516037 - REMOVAL TYPE SET AS edge
Pe�6,-908289831 | INFO | 2516037 - Caching System: False.
8]�6,-908289803 | INFO | 2516037 - Instantiating: torch_geometric.datasets.Planetoid
0i��,-908289683 | INFO | 2516037 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
�x�,-908289682 | INFO | 2516037 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
8]�6,-908289646 | INFO | 2516037 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0��S,-908289596 | INFO | 2516037 - ['all', 'all_shuffled', '-']
8]�6,-908289595 | INFO | 2516037 - Insta

In [527]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

8]�6,-908289495 | INFO | 2516037 - Instantiating: erasure.model.graphs.GCN.GCN
8]�6,-908289483 | INFO | 2516037 - Instantiating: torch.optim.Adam
8]�6,-908289482 | INFO | 2516037 - Instantiating: torch.nn.CrossEntropyLoss


graph has edges  torch.Size([2, 9104])
��t,-908289414 | INFO | 2516037 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,-908289357 | INFO | 2516037 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,-908289305 | INFO | 2516037 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,-908289253 | INFO | 2516037 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,-908289190 | INFO | 2516037 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,-908289139 | INFO | 2516037 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,-908289088 | INFO | 2516037 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph has edges  torch.Size([2, 9104])
��t,-908289036 | INFO | 2516037 - epoch = 7 ---> loss = 1.4527	 accuracy = 0.7378
graph has edges  torch.Size([2, 

In [528]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [529]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [530]:
import torch
print(torch.version.cuda)

12.6


In [531]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [532]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [533]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

0i��,-908286206 | INFO | 2516037 - Created Configurable: erasure.unlearners.certified_graph_unlearners.CEU.CEU
graph tensor([[2899, 2899, 2899,  ..., 3324, 3325, 3326],
        [ 927, 1068, 1094,  ..., 2820, 1643,   33]])
0i��,-908286092 | INFO | 2516037 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
0i��,-908286056 | INFO | 2516037 - Created Configurable: erasure.unlearners.composite.Identity


In [534]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [535]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [536]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [537]:
print(len( data_manager.partitions['forget']) )

8193


In [538]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

0i��,-908285633 | INFO | 2516037 - Created Configurable: erasure.evaluations.running.RunTime
0{��,-908285632 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-908285628 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�}��,-908285623 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-908285622 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�,-908285621 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-908285619 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�ϥ,-908285616 | INFO | 2516037 - Function: <function accuracy_score at 0x7f0ca5ded310>
0i��,-908285615 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0i��,-908285585 | INFO | 2516037 - Created Configurable: erasure.evaluations.measures.SaveValues
0i��,-908285584 | INFO

���+�,-908285553 | INFO | 2516037 - ####		 Evaluating Unlearner CEU 		####
�	��,-908285518 | INFO | 2516037 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca57f7fa0>
���S,-908285491 | INFO | 2516037 - Starting CEU (Certified Edge Unlearning)


/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:76: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)
/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:187: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  edge_index = torch.tensor(all_edges, device=device).t()
/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/CEU.py:361: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor( self.labels[self.

�	��,-908219305 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.7507507507507507 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca57f7fa0>
�	��,-908219265 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: True: 0.7537537537537538 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0c9d024bb0>
�	��,-908219236 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: False: 0.7597597597597597 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0ca57f7fa0>
�	��,-908219204 | INFO | 2516037 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: False: 0.7702702702702703 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f0c9d024bb0>
X�,-908219178 | INFO | 2516037 - ####		 Evaluating Unlearner GoldM